# Grafos em Python
## Representações, Complexidades e Graus

Prof. Lucas Nunes Alegre e Claude Sonnet 4.6

Este notebook apresenta duas formas de representar grafos em Python:

| Representação | Estrutura | Espaço |
|---|---|---|
| **Matriz de Adjacência** | `list[list[int]]` (V×V) | O(V²) |
| **Lista de Adjacência** | `dict[int, list[int]]` | O(V + E) |

**Notação:** V = número de vértices, E = número de arestas

---
## 1. Grafo com Matriz de Adjacência

Usa uma matriz **V×V** onde `matrix[i][j] = 1` indica a existência da aresta `(i → j)`.

```
     0  1  2  3
  0 [0, 1, 1, 0]
  1 [1, 0, 1, 1]
  2 [1, 1, 0, 1]
  3 [0, 1, 1, 0]
```

In [ ]:
class GrafoMatriz:
    """Grafo (dirigido ou não) usando matriz de adjacência."""

    def __init__(self, num_vertices: int, dirigido: bool = False):
        self.V = num_vertices
        self.dirigido = dirigido
        # Cria matriz V×V preenchida com 0 — O(V²)
        self.matriz = [[0] * num_vertices for _ in range(num_vertices)]

    def adicionar_aresta(self, u: int, v: int, peso: int = 1):
        """
        Marca matrix[u][v] = peso.
        Complexidade: O(1)
        """
        self.matriz[u][v] = peso
        if not self.dirigido:
            self.matriz[v][u] = peso

    def remover_aresta(self, u: int, v: int):
        """
        Zera as posições da aresta.
        Complexidade: O(1)
        """
        self.matriz[u][v] = 0
        if not self.dirigido:
            self.matriz[v][u] = 0

    def tem_aresta(self, u: int, v: int) -> bool:
        """
        Verifica se existe aresta entre u e v.
        Complexidade: O(1)
        """
        return self.matriz[u][v] != 0

    def vizinhos(self, u: int) -> list:
        """
        Retorna a lista de vértices adjacentes a u.
        Percorre a linha u da matriz procurando valores != 0.
        Complexidade: O(V)
        """
        return [v for v in range(self.V) if self.matriz[u][v] != 0]

    def grau_saida(self, u: int) -> int:
        """
        Conta arestas saindo do nodo u (linha u).
        Para grafos não-dirigidos, este é o grau do nodo.
        Complexidade: O(V)
        """
        return sum(1 for v in range(self.V) if self.matriz[u][v] != 0)

    def grau_entrada(self, u: int) -> int:
        """
        Conta arestas chegando no nodo u (coluna u).
        Relevante apenas para grafos dirigidos.
        Complexidade: O(V)
        """
        return sum(1 for v in range(self.V) if self.matriz[v][u] != 0)

    def grau_maximo(self) -> int:
        """Grau máximo entre todos os nodos. Complexidade: O(V²)"""
        return max(self.grau_saida(u) for u in range(self.V))

    def grau_minimo(self) -> int:
        """Grau mínimo entre todos os nodos. Complexidade: O(V²)"""
        return min(self.grau_saida(u) for u in range(self.V))

    def soma_graus(self) -> int:
        """
        Soma dos graus = 2 × |E|  (Handshaking Lemma).
        Complexidade: O(V²)
        """
        return sum(self.grau_saida(u) for u in range(self.V))

    def exibir(self):
        print(f"Matriz de Adjacência ({self.V} vértices):")
        print("    ", end="")
        for j in range(self.V):
            print(f" {j} ", end="")
        print()
        for i, linha in enumerate(self.matriz):
            print(f"  {i} |", end="")
            for val in linha:
                print(f" {val} ", end="")
            print()

### Demonstração — Matriz de Adjacência (Grafo não-dirigido)

Grafo de exemplo:
```
  0 --- 1
  |   / |
  |  /  |
  2 --- 3
```

In [ ]:
g = GrafoMatriz(num_vertices=4, dirigido=False)

# Adicionar arestas — O(1) cada
for u, v in [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3)]:
    g.adicionar_aresta(u, v)

g.exibir()

In [ ]:
# Verificar arestas — O(1)
print(f"(0,1) existe? {g.tem_aresta(0, 1)}")
print(f"(0,3) existe? {g.tem_aresta(0, 3)}")

In [ ]:
# Vizinhos de cada nodo — O(V) por nodo
for u in range(4):
    print(f"vizinhos({u}) = {g.vizinhos(u)}")

In [ ]:
# Grau de cada nodo — O(V) por nodo
for u in range(4):
    print(f"grau({u}) = {g.grau_saida(u)}")

print(f"\nGrau máximo : {g.grau_maximo()}  — O(V²)")
print(f"Grau mínimo : {g.grau_minimo()}  — O(V²)")
print(f"Soma graus  : {g.soma_graus()}  — O(V²)  [= 2×|E| = 2×5 = 10]")

In [ ]:
# Remover aresta — O(1)
g.remover_aresta(1, 2)
print("Após remover aresta (1,2):")
g.exibir()
print(f"\ngrau(1) após remoção: {g.grau_saida(1)}")
print(f"vizinhos(1) após remoção: {g.vizinhos(1)}")

---
## 2. Grafo com Lista de Adjacência

Cada vértice é uma chave do dicionário; o valor é uma **lista** dos vizinhos.

```python
{
  0: [1, 2],
  1: [0, 2, 3],
  2: [0, 1, 3],
  3: [1, 2]
}
```

> **Nota sobre listas vs. sets:**  
> Com `list`, a busca de vizinhos é **O(grau)** — percorre a lista linearmente.  
> Com `set`, seria **O(1)** amortizado, mas perde a ordem de inserção.

In [ ]:
class GrafoListaAdjacencia:
    """Grafo (dirigido ou não) usando dicionário de listas (lista de adjacência)."""

    def __init__(self, dirigido: bool = False):
        self.dirigido = dirigido
        # dict[int, list[int]] — O(1) para criar
        self.adj: dict[int, list] = {}

    def adicionar_vertice(self, u: int):
        """
        Insere novo vértice sem vizinhos.
        Complexidade: O(1)
        """
        if u not in self.adj:
            self.adj[u] = []

    def remover_vertice(self, u: int):
        """
        Remove o vértice e todas as arestas relacionadas a ele.
        Percorre todos os vizinhos para remover referências.
        Complexidade: O(V + E)
        """
        if u in self.adj:
            for vizinho in list(self.adj[u]):
                if vizinho in self.adj:
                    self.adj[vizinho] = [x for x in self.adj[vizinho] if x != u]
            del self.adj[u]

    def adicionar_aresta(self, u: int, v: int):
        """
        Insere v na lista de vizinhos de u (e vice-versa se não-dirigido).
        Verifica duplicatas antes de inserir: O(grau(u)).
        Complexidade: O(grau)
        """
        self.adicionar_vertice(u)
        self.adicionar_vertice(v)
        if v not in self.adj[u]:       # busca linear na lista
            self.adj[u].append(v)
        if not self.dirigido:
            if u not in self.adj[v]:
                self.adj[v].append(u)

    def remover_aresta(self, u: int, v: int):
        """
        Remove v da lista de vizinhos de u.
        list.remove() percorre a lista linearmente.
        Complexidade: O(grau)
        """
        if u in self.adj and v in self.adj[u]:
            self.adj[u].remove(v)
        if not self.dirigido and v in self.adj and u in self.adj[v]:
            self.adj[v].remove(u)

    def tem_aresta(self, u: int, v: int) -> bool:
        """
        Verifica se existe aresta entre u e v.
        Busca linear na lista de vizinhos.
        Complexidade: O(grau)
        """
        return u in self.adj and v in self.adj[u]

    def vizinhos(self, u: int) -> list:
        """
        Retorna a lista de vértices adjacentes a u.
        Acesso direto à lista — retorna uma cópia para evitar modificações externas.
        Complexidade: O(grau)  [pela cópia; O(1) se retornar referência direta]
        """
        return list(self.adj.get(u, []))

    def grau_saida(self, u: int) -> int:
        """
        Número de vizinhos de u. len() de lista é O(1).
        Para grafos não-dirigidos, este é o grau do nodo.
        Complexidade: O(1)
        """
        return len(self.adj.get(u, []))

    def grau_entrada(self, u: int) -> int:
        """
        Conta quantos nodos têm u como vizinho.
        Precisa varrer todas as listas.
        Complexidade: O(V + E)
        """
        return sum(1 for v in self.adj if u in self.adj[v])

    def grau_maximo(self) -> int:
        """Grau máximo entre todos os nodos. Complexidade: O(V)"""
        return max(len(vizinhos) for vizinhos in self.adj.values())

    def grau_minimo(self) -> int:
        """Grau mínimo entre todos os nodos. Complexidade: O(V)"""
        return min(len(vizinhos) for vizinhos in self.adj.values())

    def soma_graus(self) -> int:
        """
        Soma dos graus = 2 × |E|  (Handshaking Lemma).
        Complexidade: O(V)
        """
        return sum(len(vizinhos) for vizinhos in self.adj.values())

    def exibir(self):
        print(f"Lista de Adjacência ({len(self.adj)} vértices):")
        for v in sorted(self.adj):
            print(f"  {v} → {self.adj[v]}")

### Demonstração — Lista de Adjacência (Grafo não-dirigido)

In [ ]:
g = GrafoListaAdjacencia(dirigido=False)

# Adicionar arestas — O(grau) cada (verifica duplicatas)
for u, v in [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3)]:
    g.adicionar_aresta(u, v)

g.exibir()

In [ ]:
# Verificar arestas — O(grau) com lista
print(f"(0,1) existe? {g.tem_aresta(0, 1)}")
print(f"(0,3) existe? {g.tem_aresta(0, 3)}")

In [ ]:
# Vizinhos de cada nodo — O(grau) (cópia da lista)
for u in sorted(g.adj):
    print(f"vizinhos({u}) = {g.vizinhos(u)}")

In [ ]:
# Grau de cada nodo — O(1) por nodo (len da lista)
for u in sorted(g.adj):
    print(f"grau({u}) = {g.grau_saida(u)}")

print(f"\nGrau máximo : {g.grau_maximo()}  — O(V)")
print(f"Grau mínimo : {g.grau_minimo()}  — O(V)")
print(f"Soma graus  : {g.soma_graus()}  — O(V)  [= 2×|E| = 2×5 = 10]")

In [ ]:
# Adicionar vértice isolado — O(1)
g.adicionar_vertice(4)
print("Após adicionar vértice isolado 4:")
g.exibir()

In [ ]:
# Remover vértice — O(V + E)
g.remover_vertice(2)
print("Após remover vértice 2 (e suas arestas):")
g.exibir()

### Demonstração — Lista de Adjacência (Grafo dirigido)

```
  0 → 1 → 3
  ↓ ↗
  2
```

In [ ]:
gd = GrafoListaAdjacencia(dirigido=True)

for u, v in [(0, 1), (0, 2), (2, 1), (1, 3)]:
    gd.adicionar_aresta(u, v)

gd.exibir()

print("\nVizinhos (saída) e graus de cada nodo:")
for u in sorted(gd.adj):
    viz = gd.vizinhos(u)         # O(grau)
    gs  = gd.grau_saida(u)      # O(1)
    ge  = gd.grau_entrada(u)    # O(V + E)
    print(f"  nodo {u}: vizinhos={viz}  saída={gs}  entrada={ge}")

---
## 3. Resumo de Complexidades

| Operação | Matriz de Adjacência | Lista de Adjacência (list) |
|---|---|---|
| **Espaço** | O(V²) | O(V + E) |
| **Adicionar vértice** | O(V²) | O(1) |
| **Remover vértice** | O(V²) | O(V + E) |
| **Adicionar aresta** | O(1) | O(grau) |
| **Remover aresta** | O(1) | O(grau) |
| **Verificar aresta** | O(1) | O(grau) |
| **Vizinhos de um nodo** | O(V) | O(grau) |
| **Grau de um nodo** | O(V) | O(1) |
| **Grau máx/mín do grafo** | O(V²) | O(V) |

> **Por que O(grau) com lista?**  
> A lista não permite busca direta — é preciso percorrê-la linearmente.  
> Usar `set` tornaria `tem_aresta` e `adicionar_aresta` O(1) amortizado, ao custo de perder a ordem de inserção.

### Quando usar cada representação?

| Situação | Indicado |
|---|---|
| Grafo **denso** (E ≈ V²) | Matriz |
| Verificação de aresta muito frequente | Matriz |
| Grafo com **pesos** simples | Matriz |
| Grafo **esparso** (E << V²) | Lista |
| Grafo **dinâmico** (vértices/arestas mudam) | Lista |
| Memória é crítica | Lista |